<a href="https://colab.research.google.com/github/ShamGaneshan2008/.ipynb-files-/blob/main/Multi_Agent_Threat_Intelligence.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
!pip install pandas numpy scikit-learn xgboost -q

In [23]:
import pandas as pd

url = "https://raw.githubusercontent.com/jmnwong/NSL-KDD-Dataset/master/KDDTrain%2B.txt"

data = pd.read_csv(url, header=None)

print("Shape: ", data.shape)
data.head()


Shape:  (125973, 43)


,0,1,2,3,4,5,6,7,8,9,...,33,34,35,36,37,38,39,40,41,42
0,0,tcp,ftp_data,SF,491,0,0,0,0,0,...,0.17,0.03,0.17,0.00,0.00,0.00,0.05,0.00,normal,20
1,0,udp,other,SF,146,0,0,0,0,0,...,0.00,0.60,0.88,0.00,0.00,0.00,0.00,0.00,normal,15
2,0,tcp,private,S0,0,0,0,0,0,0,...,0.10,0.05,0.00,0.00,1.00,1.00,0.00,0.00,neptune,19
3,0,tcp,http,SF,232,8153,0,0,0,0,...,1.00,0.00,0.03,0.04,0.03,0.01,0.00,0.01,normal,21
4,0,tcp,http,SF,199,420,0,0,0,0,...,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,normal,21


In [24]:
columns = [
    "duration", "protocol_type", "service", "flag", "src_bytes",
    "dst_bytes", "land", "wrong_fragment", "urgent", "hot",
    "num_failed_logins", "logged_in", "num_compromised",
    "root_shell", "su_attempted", "num_root", "num_file_creations",
    "num_shells", "num_access_files", "num_outbound_cmds",
    "is_host_login", "is_guest_login", "count", "srv_count",
    "serror_rate", "srv_serror_rate", "rerror_rate",
    "srv_rerror_rate", "same_srv_rate", "diff_srv_rate",
    "srv_diff_host_rate", "dst_host_count", "dst_host_srv_count",
    "dst_host_same_srv_rate", "dst_host_diff_srv_rate",
    "dst_host_same_src_port_rate", "dst_host_srv_diff_host_rate",
    "dst_host_serror_rate", "dst_host_srv_serror_rate",
    "dst_host_rerror_rate", "dst_host_srv_rerror_rate",
    "attack", "difficulty"
]

data.columns = columns

print(data.shape)
data.head()

print(data["attack"].value_counts())

(125973, 43)
attack
normal             67343
neptune            41214
satan               3633
ipsweep             3599
portsweep           2931
smurf               2646
nmap                1493
back                 956
teardrop             892
warezclient          890
pod                  201
guess_passwd          53
buffer_overflow       30
warezmaster           20
land                  18
imap                  11
rootkit               10
loadmodule             9
ftp_write              8
multihop               7
phf                    4
perl                   3
spy                    2
Name: count, dtype: int64


In [25]:
data["target"] = data["attack"].apply(
    lambda x: 0 if x == "normal" else 1
)

print(data["target"].value_counts())

data[["target","attack"]].head(10)

target
0    67343
1    58630
Name: count, dtype: int64


,target,attack
0,0,normal
1,0,normal
2,1,neptune
3,0,normal
4,0,normal
5,1,neptune
6,1,neptune
7,1,neptune
8,1,neptune
9,1,neptune


In [29]:
from sklearn.model_selection import train_test_split

X = data.drop(columns=["attack","target","difficulty"])
y = data["target"]

X = pd.get_dummies(X, columns=["protocol_type", "service", "flag"])

X_train,X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

print("X_train", X_train.shape)
print("X_test", X_test.shape)
print("y_train", y_train.shape)
print("y_test", y_test.shape)

X_train (88181, 122)
X_test (37792, 122)
y_train (88181,)
y_test (37792,)


In [30]:
from xgboost import XGBClassifier

behavior_features = [
    "num_failed_logins",
    "num_compromised",
    "root_shell",
    "su_attempted",
    "num_root",
    "num_file_creations",
    "num_shells",
    "num_access_files",
    "is_host_login",
    "is_guest_login",
    "logged_in",
    "hot"
]

behavior_features = [f for f in behavior_features if f in X_train.columns]

behavior_model = XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    random_state=42
)

behavior_model.fit(
    X_train[behavior_features],
    y_train
)

print("Behavior Agent trained")

Behavior Agent trained


In [31]:
behavior_pred = behavior_model.predict(X_test[behavior_features])

print(behavior_pred[:20])

[1 1 0 0 1 0 0 0 0 0 1 0 1 0 0 1 1 0 0 0]


In [32]:
from sklearn.ensemble import RandomForestClassifier

network_features = [
    "duration",
    "src_bytes",
    "dst_bytes",
    "count",
    "srv_count",
    "serror_rate",
    "srv_serror_rate",
    "rerror_rate",
    "srv_rerror_rate",
    "same_srv_rate",
    "diff_srv_rate",
    "dst_host_count",
    "dst_host_srv_count",
    "dst_host_same_srv_rate",
    "dst_host_diff_srv_rate"
]

network_features = [f for f in network_features if f in X_train.columns]

network_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

network_model.fit(
    X_train[network_features],
    y_train
)

print("Network Agent trained")

Network Agent trained


In [33]:
network_pred = network_model.predict(X_test[network_features])

print(network_pred[:20])

[0 1 0 0 0 0 0 0 0 0 1 0 1 0 0 1 0 0 0 0]


In [35]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

sequence_features = [
    "duration",
    "src_bytes", # Corrected from 'src_byte'
    "dst_bytes", # Corrected from 'dst_byte'
    "count",
    "srv_count"
]

sequence_data = data[sequence_features].values
sequence_target = data["target"].values

window = 5

X_seq = []
y_seq = []

for i in range(window, len(sequence_data)):
  X_seq.append(sequence_data[i - window:i])
  y_seq.append(sequence_target[i])


X_seq = np.array(X_seq)
y_seq = np.array(y_seq)

split_seq = int(len(X_seq) * 0.8)

X_seq_train = X_seq[:split_seq]
X_seq_test = X_seq[split_seq:]

y_seq_train = y_seq[:split_seq]
y_seq_test = y_seq[split_seq:]

print("Trianing: ", X_seq_train.shape)
print("Testing: ", X_seq_test.shape)

Trianing:  (100774, 5, 5)
Testing:  (25194, 5, 5)


In [36]:
temporal_model = Sequential([
    LSTM(32, input_shape=(window, len(sequence_features))),
    Dense(1, activation="sigmoid")
])

temporal_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

temporal_model.fit(
    X_seq_train,
    y_seq_train,
    epochs=5,
    batch_size=64,
    validation_split=0.1,
    verbose=1
)

print("Temporal Agent trained!")

/usr/local/lib/python3.13/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/5
1418/1418 ━━━━━━━━━━━━━━━━━━━━ 9s 4ms/step - accuracy: 0.5237 - loss: 0.6936 - val_accuracy: 0.5392 - val_loss: 0.6907
Epoch 2/5
1418/1418 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - accuracy: 0.5327 - loss: 0.6913 - val_accuracy: 0.5395 - val_loss: 0.6903
Epoch 3/5
1418/1418 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.5340 - loss: 0.6911 - val_accuracy: 0.5393 - val_loss: 0.6903
Epoch 4/5
1418/1418 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - accuracy: 0.5337 - loss: 0.6910 - val_accuracy: 0.5393 - val_loss: 0.6903
Epoch 5/5
1418/1418 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - accuracy: 0.5338 - loss: 0.6910 - val_accuracy: 0.5385 - val_loss: 0.6911
Temporal Agent trained!


In [37]:
temporal_prob = temporal_model.predict(X_seq_test, verbose=0).flatten()

temporal_pred = (temporal_prob >= 0.5).astype(int)

print(temporal_pred[:20])

[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]


In [38]:
test_start = split_seq + window

X_agent_train = X.iloc[:test_start]
X_agent_test = X.iloc[test_start:]

y_agent_train = y.iloc[:test_start]
y_agent_test = y.iloc[test_start:]

behavior_model.fit(
    X_agent_train[behavior_features],
    y_agent_train
)

network_model.fit(
    X_agent_train[network_features],
    y_agent_train
)

behavior_prob = behavior_model.predict_proba(
    X_agent_test[behavior_features]
)[:, 1]

network_prob = network_model.predict_proba(
    X_agent_test[network_features]
)[:, 1]

print("Behavior predictions:", len(behavior_prob))
print("Network predictions :", len(network_prob))
print("Temporal predictions:", len(temporal_prob))

Behavior predictions: 25194
Network predictions : 25194
Temporal predictions: 25194


In [39]:
fusion_data = pd.DataFrame({
    "behavior": behavior_prob,
    "network": network_prob,
    "temporal": temporal_prob
})

fusion_target = y_agent_test.iloc[-len(fusion_data):].values

print(fusion_data.head())
print("Fusion data:", fusion_data.shape)

   behavior  network  temporal
0  0.014549      0.0  0.481184
1  0.014549      0.0  0.482843
2  0.743107      1.0  0.482185
3  0.014549      0.0  0.472827
4  0.743107      1.0  0.479613
Fusion data: (25194, 3)


In [40]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

split_fusion = int(len(fusion_data) * 0.8)

fusion_X_train  = fusion_data.iloc[:split_fusion]
fusion_X_test = fusion_data.iloc[split_fusion:]

fusion_y_train  = fusion_target[:split_fusion]
fusion_y_test = fusion_target[split_fusion:]

fusion_model = LogisticRegression()

fusion_model.fit(
    fusion_X_train,
    fusion_y_train
)

print("Fusion Agent trained!")

Fusion Agent trained!


In [41]:
final_pred = fusion_model.predict(fusion_X_test)

accuracy = accuracy_score(
    fusion_y_test,
    final_pred
)

print(f"AEGIS Accuracy: {accuracy:.2%}")

print("\nClassification Report:")
print(
    classification_report(
        fusion_y_test,
        final_pred,
        target_names=["NORMAL", "ATTACK"],
        zero_division=0
    )
)

AEGIS Accuracy: 99.80%

Classification Report:
              precision    recall  f1-score   support

      NORMAL       1.00      1.00      1.00      2703
      ATTACK       1.00      1.00      1.00      2336

    accuracy                           1.00      5039
   macro avg       1.00      1.00      1.00      5039
weighted avg       1.00      1.00      1.00      5039



In [42]:
final_prob = fusion_model.predict_proba(fusion_X_test)[:, 1]

investigation = fusion_X_test.copy()

investigation["AEGIS_Score"] = final_prob
investigation["Prediction"] = np.where(
    final_prob >= 0.5,
    "ATTACK",
    "NORMAL"
)

investigation.head(10)

,behavior,network,temporal,AEGIS_Score,Prediction
20155,0.014549,0.00,0.483250,0.001406,NORMAL
20156,0.014549,0.00,0.463296,0.001463,NORMAL
20157,0.743107,1.00,0.462526,0.998588,ATTACK
20158,0.743107,0.00,0.467527,0.003779,NORMAL
20159,0.014549,0.01,0.470599,0.001627,NORMAL
20160,0.743107,0.03,0.486448,0.005227,NORMAL
20161,0.981448,1.00,0.459213,0.998975,ATTACK
20162,0.743107,1.00,0.471815,0.998562,ATTACK
20163,0.014549,0.00,0.477279,0.001422,NORMAL
20164,0.743107,1.00,0.485483,0.998522,ATTACK


In [43]:
import plotly.graph_objects as go

sample = investigation.head(20)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=list(range(len(sample))),
    y=sample["behavior"],
    mode="lines+markers",
    name="Behavior Agent"
))

fig.add_trace(go.Scatter(
    x=list(range(len(sample))),
    y=sample["network"],
    mode="lines+markers",
    name="Network Agent"
))

fig.add_trace(go.Scatter(
    x=list(range(len(sample))),
    y=sample["temporal"],
    mode="lines+markers",
    name="Temporal Agent"
))

fig.add_trace(go.Scatter(
    x=list(range(len(sample))),
    y=sample["AEGIS_Score"],
    mode="lines+markers",
    name="AEGIS Score"
))

fig.add_hline(
    y=0.5,
    line_dash="dash"
)

fig.update_layout(
    title="AEGIS Multi-Agent Threat Dashboard",
    xaxis_title="Event",
    yaxis_title="Threat Probability",
    yaxis=dict(range=[0, 1]),
    template="plotly_white"
)

fig.show()

In [44]:
event_indices = X_agent_test.index[split_fusion:]

risk_table = data.loc[event_indices, [
    "duration",
    "protocol_type",
    "service",
    "src_bytes",
    "dst_bytes",
    "attack"
]].copy()

risk_table["Behavior"] = behavior_prob[split_fusion:]
risk_table["Network"] = network_prob[split_fusion:]
risk_table["Temporal"] = temporal_prob[split_fusion:]
risk_table["AEGIS_Score"] = final_prob

risk_table["Risk"] = np.where(
    risk_table["AEGIS_Score"] >= 0.8,
    "HIGH",
    np.where(
        risk_table["AEGIS_Score"] >= 0.5,
        "MEDIUM",
        "LOW"
    )
)

risk_table = risk_table.sort_values(
    "AEGIS_Score",
    ascending=False
)

risk_table.head(10)

,duration,protocol_type,service,src_bytes,dst_bytes,attack,Behavior,Network,Temporal,AEGIS_Score,Risk
125065,0,tcp,telnet,126,179,guess_passwd,0.983389,1.0,0.455697,0.998985,HIGH
120940,0,tcp,http,54540,8314,back,0.981448,1.0,0.459213,0.998975,HIGH
122051,0,tcp,http,54540,8314,back,0.981448,1.0,0.460468,0.998973,HIGH
124878,1,tcp,ftp,1216,2449,warezclient,0.972998,1.0,0.455481,0.998971,HIGH
124825,0,tcp,telnet,126,179,guess_passwd,0.983389,1.0,0.463111,0.998970,HIGH
125736,0,tcp,http,54540,8314,back,0.981448,1.0,0.462427,0.998969,HIGH
125929,0,tcp,http,54540,8314,back,0.981448,1.0,0.464748,0.998964,HIGH
123833,0,tcp,http,54540,8314,back,0.981448,1.0,0.466520,0.998960,HIGH
125877,0,tcp,http,54540,8314,back,0.981448,1.0,0.467214,0.998959,HIGH
121872,0,tcp,http,54540,8314,back,0.981448,1.0,0.467295,0.998958,HIGH


In [45]:
behavior_importance = pd.Series(
    behavior_model.feature_importances_,
    index=behavior_features
).sort_values(ascending=False)

network_importance = pd.Series(
    network_model.feature_importances_,
    index=network_features
).sort_values(ascending=False)

print("Top Behavior Features:")
print(behavior_importance.head(5))

print("\nTop Network Features:")
print(network_importance.head(5))

Top Behavior Features:
logged_in          0.973255
hot                0.011987
num_compromised    0.009646
is_guest_login     0.002055
root_shell         0.000868
dtype: float32

Top Network Features:
src_bytes                 0.267919
dst_bytes                 0.186660
dst_host_same_srv_rate    0.091266
diff_srv_rate             0.087257
dst_host_srv_count        0.078149
dtype: float64


In [46]:
top_behavior = behavior_importance.head(3).index.tolist()
top_network = network_importance.head(3).index.tolist()

print("AEGIS Explanation")
print("")
print("Behavior Agent focuses on:", ", ".join(top_behavior))
print("Network Agent focuses on :", ", ".join(top_network))
print("Temporal Agent focuses on event sequences")

AEGIS Explanation

Behavior Agent focuses on: logged_in, hot, num_compromised
Network Agent focuses on : src_bytes, dst_bytes, dst_host_same_srv_rate
Temporal Agent focuses on event sequences


In [47]:
event = risk_table.iloc[0]

print("========== AEGIS INVESTIGATION ==========")

print("\nEvent Details")
print("----------------")
print("Service :", event["service"])
print("Protocol:", event["protocol_type"])
print("Source Bytes:", event["src_bytes"])
print("Destination Bytes:", event["dst_bytes"])

print("\nAgent Analysis")
print("----------------")
print(f"Behavior Agent : {event['Behavior']:.2%}")
print(f"Network Agent  : {event['Network']:.2%}")
print(f"Temporal Agent : {event['Temporal']:.2%}")

print("\nFinal Assessment")
print("----------------")
print(f"AEGIS Score: {event['AEGIS_Score']:.2%}")
print(f"Risk Level : {event['Risk']}")

print("\nOriginal Label")
print("----------------")
print(event["attack"])

========== AEGIS INVESTIGATION ==========

Event Details
----------------
Service : telnet
Protocol: tcp
Source Bytes: 126
Destination Bytes: 179

Agent Analysis
----------------
Behavior Agent : 98.34%
Network Agent  : 100.00%
Temporal Agent : 45.57%

Final Assessment
----------------
AEGIS Score: 99.90%
Risk Level : HIGH

Original Label
----------------
guess_passwd


In [51]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

event = risk_table.iloc[0]

behavior = float(event["Behavior"])
network = float(event["Network"])
temporal = float(event["Temporal"])
threat = float(event["AEGIS_Score"])

risk = str(event["Risk"])

if risk == "HIGH":
    risk_color = "#ff3b30"
elif risk == "MEDIUM":
    risk_color = "#ffb020"
else:
    risk_color = "#00e676"

fig = make_subplots(
    rows=3,
    cols=3,
    specs=[
        [{"type": "indicator", "colspan": 2}, None, {"type": "indicator"}],
        [{"type": "xy"}, {"type": "xy"}, {"type": "xy"}],
        [{"type": "table", "colspan": 3}, None, None]
    ],
    row_heights=[0.38, 0.30, 0.32],
    vertical_spacing=0.08,
    horizontal_spacing=0.05
)

# =========================
# MAIN THREAT GAUGE
# =========================

fig.add_trace(
    go.Indicator(
        mode="gauge+number",
        value=threat * 100,
        number={
            "suffix": "%",
            "font": {
                "size": 48,
                "color": "white"
            }
        },
        title={
            "text": "<b>AEGIS THREAT SCORE</b>",
            "font": {
                "size": 18,
                "color": "#cbd5e1"
            }
        },
        gauge={
            "axis": {
                "range": [0, 100],
                "tickfont": {
                    "color": "#94a3b8"
                }
            },
            "bar": {
                "color": risk_color,
                "thickness": 0.28
            },
            "bgcolor": "#0f172a",
            "borderwidth": 2,
            "bordercolor": "#334155",
            "steps": [
                {
                    "range": [0, 50],
                    "color": "#172033"
                },
                {
                    "range": [50, 80],
                    "color": "#222b3d"
                },
                {
                    "range": [80, 100],
                    "color": "#2c2530"
                }
            ]
        }
    ),
    row=1,
    col=1
)

# =========================
# RISK CARD
# =========================

fig.add_trace(
    go.Indicator(
        mode="number",
        value=0,
        title={
            "text": (
                f"<b style='color:{risk_color};font-size:34px'>"
                f"{risk}"
                f"</b>"
                "<br><span style='font-size:13px;color:#94a3b8'>"
                "CURRENT RISK"
                "</span>"
            )
        },
        number={
            "font": {
                "size": 1,
                "color": "rgba(0,0,0,0)"
            }
        }
    ),
    row=1,
    col=3
)

# =========================
# BEHAVIOR AGENT
# =========================

fig.add_trace(
    go.Bar(
        x=[behavior],
        y=[""],
        orientation="h",
        text=[f"{behavior:.1%}"],
        textposition="inside",
        insidetextanchor="middle",
        marker=dict(
            color="#00d4ff",
            line=dict(width=0)
        ),
        hovertemplate="Behavior Agent: %{x:.2%}<extra></extra>",
        showlegend=False
    ),
    row=2,
    col=1
)

# =========================
# NETWORK AGENT
# =========================

fig.add_trace(
    go.Bar(
        x=[network],
        y=[""],
        orientation="h",
        text=[f"{network:.1%}"],
        textposition="inside",
        insidetextanchor="middle",
        marker=dict(
            color="#a855f7",
            line=dict(width=0)
        ),
        hovertemplate="Network Agent: %{x:.2%}<extra></extra>",
        showlegend=False
    ),
    row=2,
    col=2
)

# =========================
# TEMPORAL AGENT
# =========================

fig.add_trace(
    go.Bar(
        x=[temporal],
        y=[""],
        orientation="h",
        text=[f"{temporal:.1%}"],
        textposition="inside",
        insidetextanchor="middle",
        marker=dict(
            color="#00e676",
            line=dict(width=0)
        ),
        hovertemplate="Temporal Agent: %{x:.2%}<extra></extra>",
        showlegend=False
    ),
    row=2,
    col=3
)

# =========================
# INVESTIGATION TABLE
# =========================

fig.add_trace(
    go.Table(
        header=dict(
            values=[
                "<b>RISK</b>",
                "<b>AEGIS SCORE</b>",
                "<b>SERVICE</b>",
                "<b>PROTOCOL</b>",
                "<b>SOURCE BYTES</b>",
                "<b>DESTINATION BYTES</b>"
            ],
            fill_color="#172033",
            line_color="#334155",
            font=dict(
                color="white",
                size=13
            ),
            align="center",
            height=40
        ),
        cells=dict(
            values=[
                [risk],
                [f"{threat:.2%}"],
                [str(event["service"])],
                [str(event["protocol_type"])],
                [f"{int(event['src_bytes']):,}"],
                [f"{int(event['dst_bytes']):,}"]
            ],
            fill_color="#0f172a",
            line_color="#334155",
            font=dict(
                color="#e2e8f0",
                size=12
            ),
            align="center",
            height=42
        )
    ),
    row=3,
    col=1
)

# =========================
# AGENT TITLES
# =========================

fig.add_annotation(
    x=0.165,
    y=0.325,
    xref="paper",
    yref="paper",
    text="<b>BEHAVIOR AGENT</b><br><span style='color:#00d4ff'>XGBoost</span>",
    showarrow=False,
    font=dict(size=14, color="white")
)

fig.add_annotation(
    x=0.50,
    y=0.325,
    xref="paper",
    yref="paper",
    text="<b>NETWORK AGENT</b><br><span style='color:#a855f7'>Random Forest</span>",
    showarrow=False,
    font=dict(size=14, color="white")
)

fig.add_annotation(
    x=0.835,
    y=0.325,
    xref="paper",
    yref="paper",
    text="<b>TEMPORAL AGENT</b><br><span style='color:#00e676'>LSTM</span>",
    showarrow=False,
    font=dict(size=14, color="white")
)

# =========================
# HEADER
# =========================

fig.add_annotation(
    x=0.5,
    y=1.08,
    xref="paper",
    yref="paper",
    text=(
        "🛡️ <b>AEGIS</b>"
        "<br>"
        "<span style='font-size:15px;color:#94a3b8'>"
        "MULTI-AGENT AI THREAT INTELLIGENCE CENTER"
        "</span>"
    ),
    showarrow=False,
    align="center",
    font=dict(
        size=27,
        color="white"
    )
)

# =========================
# SUBTITLE
# =========================

fig.add_annotation(
    x=0.5,
    y=1.005,
    xref="paper",
    yref="paper",
    text=(
        "BEHAVIOR • NETWORK • TEMPORAL • FUSION INTELLIGENCE"
    ),
    showarrow=False,
    font=dict(
        size=11,
        color="#64748b"
    )
)

# =========================
# AGENT CONSENSUS
# =========================

fig.add_annotation(
    x=0.5,
    y=0.395,
    xref="paper",
    yref="paper",
    text=(
        f"AGENT CONSENSUS&nbsp;&nbsp; "
        f"<b>{behavior:.0%}</b> • "
        f"<b>{network:.0%}</b> • "
        f"<b>{temporal:.0%}</b>"
    ),
    showarrow=False,
    font=dict(
        size=12,
        color="#cbd5e1"
    )
)

# =========================
# BAR SETTINGS
# =========================

for col in [1, 2, 3]:
    fig.update_xaxes(
        range=[0, 1],
        showgrid=False,
        showticklabels=False,
        zeroline=False,
        row=2,
        col=col
    )

    fig.update_yaxes(
        showgrid=False,
        showticklabels=False,
        zeroline=False,
        row=2,
        col=col
    )

# =========================
# BACKGROUND PANELS
# =========================

panel_color = "#111827"

fig.update_layout(
    height=900,
    paper_bgcolor="#050b14",
    plot_bgcolor=panel_color,
    font=dict(
        family="Arial",
        color="white"
    ),
    margin=dict(
        l=45,
        r=45,
        t=125,
        b=45
    ),
    showlegend=False
)

# =========================
# PANEL BORDERS
# =========================

fig.add_shape(
    type="rect",
    x0=0.01,
    x1=0.67,
    y0=0.61,
    y1=0.99,
    xref="paper",
    yref="paper",
    line=dict(color="#263449", width=1),
    fillcolor="#0b1220",
    layer="below"
)

fig.add_shape(
    type="rect",
    x0=0.70,
    x1=0.99,
    y0=0.61,
    y1=0.99,
    xref="paper",
    yref="paper",
    line=dict(color="#263449", width=1),
    fillcolor="#0b1220",
    layer="below"
)

fig.add_shape(
    type="rect",
    x0=0.01,
    x1=0.99,
    y0=0.29,
    y1=0.57,
    xref="paper",
    yref="paper",
    line=dict(color="#263449", width=1),
    fillcolor="#0b1220",
    layer="below"
)

fig.add_shape(
    type="rect",
    x0=0.01,
    x1=0.99,
    y0=0.01,
    y1=0.25,
    xref="paper",
    yref="paper",
    line=dict(color="#263449", width=1),
    fillcolor="#0b1220",
    layer="below"
)

fig.show()

print("\n🛡️ AEGIS INVESTIGATION COMPLETE")
print("━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print(f"Risk Level    : {risk}")
print(f"Threat Score  : {threat:.2%}")
print(f"Behavior      : {behavior:.2%}")
print(f"Network       : {network:.2%}")
print(f"Temporal      : {temporal:.2%}")
print(f"Service       : {event['service']}")
print(f"Protocol      : {event['protocol_type']}")


🛡️ AEGIS INVESTIGATION COMPLETE
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Risk Level    : HIGH
Threat Score  : 99.90%
Behavior      : 98.34%
Network       : 100.00%
Temporal      : 45.57%
Service       : telnet
Protocol      : tcp
